# Artificial Intelligence
# 464
# Project #5

## Before You Begin...
00. We're using a Jupyter Notebook environment (tutorial available here: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html),
01. Read the entire notebook before beginning your work, and
02.  Check the submission deadline on Gradescope.


## General Directions for this Assignment
00. Output format should be exactly as requested,
01. Functions should do only one thing.


## Before You Submit...
00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.

## Neural Networks: Architecture

For this assignment we will explore Neural Networks; in particular, we are going to explore model complexity. The goal is classify a mushroom as either edible ('e') or poisonous ('p'). The dataset has been uploaded to Canvas. In case you'd like to learn more about it, here's the link to the repo: https://archive.ics.uci.edu/dataset/73/mushroom. Use PyTorch (https://pytorch.org/) to explore different model complexities (i.e., architectures) before declaring a winner. Either start with a simple network and make it more complex; or start with a complex model and pare it down. Either way, your submission should clearly demonstrate your exploration. 

More specifically: 

* Try three different models, 
* Use cross validation to compare the models, 
* Report the winning model's performance on test data (this is the only time you will use test data), 
* Include a table comparing the three models, 
* Document your process: what the results were, how the winning model was determined, loss function, activation, etc.

Information on cross validation: https://scikit-learn.org/stable/modules/cross_validation.html (notice how final evaluation is the only time test data is used). No other directions for this assignment other than what's here and in the "General Directions" section. You have a lot of freedom with this assignment. Don't get carried away. It is expected the results may vary, being better or worse, due to the limitations of the dataset. Graders are not going to run your notebooks. The notebook will be read as a report on how different models were explored. Since you'll be using libraries, the emphasis will be on your ability to communicate your findings.

In [1]:
# Imports
# Import necessary libraries for data manipulation, model training, and evaluation
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [2]:
# Load and preprocess dataset
# Define column names from the dataset documentation
column_names = [
    "class","cap-shape","cap-surface","cap-color","bruises","odor","gill-attachment",
    "gill-spacing","gill-size","gill-color","stipe-shape","stipe-root",
    "stipe-surface-above-ring","stipe-surface-below-ring","stipe-color-above-ring",
    "stipe-color-below-ring","veil-type","veil-color","ring-number","ring-type",
    "spore-print-color","population","habitat"
]

In [3]:
# Load CSV and fill missing values with 'missing' category
df = pd.read_csv("agaricus-lepiota.data", names=column_names, na_values="?")
df = df.fillna("missing")

# Separate features and labels
X = df.drop("class", axis=1)
y = df["class"]

# Encode label as binary: edible = 0, poisonous = 1
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Encode each categorical feature as an integer
for col in X.columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Convert data to numpy float32 for PyTorch compatibility
X = X.astype(np.float32)
y = y.astype(np.int64)

# Train/val/test split
# 80% for training/validation, 20% held out as test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
# Dataset class
# PyTorch Dataset wrapper for loading batches
class MushroomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if hasattr(self.X, 'iloc'):
            x = self.X.iloc[idx].values
        else:
            x = self.X[idx]
            
        if hasattr(self.y, 'iloc'):
            y = self.y.iloc[idx]
        else:
            y = self.y[idx]

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)




In [5]:
# Model definitions
# Three models with increasing complexity
input_dim = X_trainval.shape[1]

class ModelA(nn.Module):
    # Simple model with one hidden layer
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, 2)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class ModelB(nn.Module):
    # Moderately complex model with 3 hidden layers
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 2)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

class ModelC(nn.Module):
    # Most complex model with 4 hidden layers and dropout
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.drop2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 2)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        return self.fc5(x)


In [6]:
# Training & evaluation routines
# Function to train one epoch and return average loss/accuracy
def train(model, optimizer, criterion, dataloader, device):
    model.train()
    total, correct, loss_total = 0, 0, 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * y_batch.size(0)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += y_batch.size(0)
    return loss_total / total, correct / total

# Function to evaluate model on validation or test set
def evaluate(model, criterion, dataloader, device):
    model.eval()
    total, correct, loss_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss_total += loss.item() * y_batch.size(0)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
    return loss_total / total, correct / total

# K-Fold Cross-Validation
# Evaluate each model using 5-fold CV
def run_kfold_cv(model_class, X_data, y_data, k=5, epochs=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Convert to NumPy arrays if not already
    if hasattr(X_data, 'values'):
        X_data = X_data.values
    if hasattr(y_data, 'values'):
        y_data = y_data.values
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data), 1):
        print(f"===== Fold {fold_idx} =====")
        model = model_class().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        train_loader = DataLoader(MushroomDataset(X_data[train_idx], y_data[train_idx]), batch_size=64, shuffle=True)
        val_loader = DataLoader(MushroomDataset(X_data[val_idx], y_data[val_idx]), batch_size=64)

        for epoch in range(epochs):
            train_loss, train_acc = train(model, optimizer, criterion, train_loader, device)
            val_loss, val_acc = evaluate(model, criterion, val_loader, device)
            print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

        _, val_acc = evaluate(model, criterion, val_loader, device)
        scores.append(val_acc)
    return np.mean(scores)

# Run CV on all three models
cv_results = {}
for model_cls, name in zip([ModelA, ModelB, ModelC], ["Model A", "Model B", "Model C"]):
    # Call site
    acc = run_kfold_cv(model_cls, X_trainval, y_trainval, epochs=30)
    cv_results[name] = acc
    print(f"{name} CV Accuracy: {acc:.4f}")


===== Fold 1 =====
Epoch 01: Train Loss=0.5422, Train Acc=0.7544, Val Loss=0.4622, Val Acc=0.8062
Epoch 02: Train Loss=0.3817, Train Acc=0.8529, Val Loss=0.3392, Val Acc=0.8754
Epoch 03: Train Loss=0.2881, Train Acc=0.8890, Val Loss=0.2674, Val Acc=0.9023
Epoch 04: Train Loss=0.2278, Train Acc=0.9133, Val Loss=0.2194, Val Acc=0.9138
Epoch 05: Train Loss=0.1876, Train Acc=0.9277, Val Loss=0.1852, Val Acc=0.9223
Epoch 06: Train Loss=0.1564, Train Acc=0.9354, Val Loss=0.1584, Val Acc=0.9346
Epoch 07: Train Loss=0.1324, Train Acc=0.9481, Val Loss=0.1364, Val Acc=0.9415
Epoch 08: Train Loss=0.1119, Train Acc=0.9583, Val Loss=0.1208, Val Acc=0.9531
Epoch 09: Train Loss=0.0968, Train Acc=0.9654, Val Loss=0.1001, Val Acc=0.9677
Epoch 10: Train Loss=0.0829, Train Acc=0.9752, Val Loss=0.0897, Val Acc=0.9662
Epoch 11: Train Loss=0.0706, Train Acc=0.9802, Val Loss=0.0761, Val Acc=0.9769
Epoch 12: Train Loss=0.0618, Train Acc=0.9865, Val Loss=0.0684, Val Acc=0.9800
Epoch 13: Train Loss=0.0527, Trai

In [7]:
# Select best model, retrain on all trainval, and test
best_model_name = max(cv_results, key=cv_results.get)
best_model_cls = {"Model A": ModelA, "Model B": ModelB, "Model C": ModelC}[best_model_name]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = best_model_cls().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

trainval_loader = DataLoader(MushroomDataset(X_trainval, y_trainval), batch_size=64, shuffle=True)
test_loader = DataLoader(MushroomDataset(X_test, y_test), batch_size=64)

# Train best model for 50 epochs on full train+val set
for epoch in range(50):
    train_loss, train_acc = train(model, optimizer, criterion, trainval_loader, device)
    val_loss, val_acc = evaluate(model, criterion, test_loader, device)
    print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Test Loss={val_loss:.4f}, Test Acc={val_acc:.4f}")

# Final performance on the test set
_, test_acc = evaluate(model, criterion, test_loader, device)
print(f"Final Test Accuracy of {best_model_name}: {test_acc:.4f}")

Epoch 01: Train Loss=0.4466, Train Acc=0.8167, Test Loss=0.2978, Test Acc=0.8831
Epoch 02: Train Loss=0.2062, Train Acc=0.9298, Test Loss=0.1553, Test Acc=0.9422
Epoch 03: Train Loss=0.0990, Train Acc=0.9649, Test Loss=0.0591, Test Acc=0.9815
Epoch 04: Train Loss=0.0373, Train Acc=0.9900, Test Loss=0.0288, Test Acc=0.9938
Epoch 05: Train Loss=0.0174, Train Acc=0.9960, Test Loss=0.0163, Test Acc=0.9963
Epoch 06: Train Loss=0.0093, Train Acc=0.9985, Test Loss=0.0068, Test Acc=1.0000
Epoch 07: Train Loss=0.0053, Train Acc=0.9998, Test Loss=0.0041, Test Acc=1.0000
Epoch 08: Train Loss=0.0033, Train Acc=0.9998, Test Loss=0.0031, Test Acc=0.9994
Epoch 09: Train Loss=0.0023, Train Acc=0.9998, Test Loss=0.0019, Test Acc=1.0000
Epoch 10: Train Loss=0.0016, Train Acc=1.0000, Test Loss=0.0020, Test Acc=1.0000
Epoch 11: Train Loss=0.0015, Train Acc=1.0000, Test Loss=0.0015, Test Acc=1.0000
Epoch 12: Train Loss=0.0017, Train Acc=1.0000, Test Loss=0.0010, Test Acc=1.0000
Epoch 13: Train Loss=0.0012,

In [8]:
# Summary Table Comparing Models
print("\nComparison of Neural Network Architectures:")
print("| Model   | Architecture Description                    | CV Accuracy |")
print("|---------|---------------------------------------------|--------------|")
print("| Model A | 1 hidden layer (16 units)                  | {:.4f}       |".format(cv_results['Model A']))
print("| Model B | 3 hidden layers (64-32-16 units)           | {:.4f}       |".format(cv_results['Model B']))
print("| Model C | 4 hidden layers (128-64-32-16) + Dropout   | {:.4f}       |".format(cv_results['Model C']))



Comparison of Neural Network Architectures:
| Model   | Architecture Description                    | CV Accuracy |
|---------|---------------------------------------------|--------------|
| Model A | 1 hidden layer (16 units)                  | 0.9992       |
| Model B | 3 hidden layers (64-32-16 units)           | 1.0000       |
| Model C | 4 hidden layers (128-64-32-16) + Dropout   | 0.9995       |


Documentation


I explored the impact of model complexity on the performance of neural networks for classifying mushrooms as either edible or poisonous. To do this, we implemented three feedforward neural network architectures of increasing complexity using PyTorch. Each model was evaluated using 5-fold cross-validation on the training and validation set. Our goal was to identify a model that balanced strong predictive performance with appropriate complexity, avoiding overfitting.

Model A was a simple network with one hidden layer containing 16 units. This served as our baseline architecture. Model B was moderately complex, consisting of three hidden layers with 64, 32, and 16 units respectively. Model C was the most complex, with four hidden layers containing 128, 64, 32, and 16 units, and dropout layers applied after the first two hidden layers to reduce overfitting. All models used the ReLU activation function between layers and were trained using the cross-entropy loss function. Optimization was performed using the Adam optimizer with a learning rate of 0.001.

During cross-validation, we trained each model for 30 epochs per fold. The performance metric used was accuracy. The results showed that all models performed very well on this dataset. Model A achieved a cross-validation accuracy of 99.92%, Model B achieved 100%, and Model C achieved 99.95%. These high accuracies suggest that the mushroom dataset is relatively easy to separate using its categorical features. In particular, certain features, such as odor, are highly predictive and likely contribute to the strong performance across all models.

Although Model B achieved perfect accuracy in cross-validation, this may reflect the high separability of the dataset rather than an indication that increased complexity is always beneficial. Model C, which was more complex and included dropout regularization, performed slightly below Model B. This suggests that while added complexity and regularization can help with generalization, they may not always lead to better results when the dataset is already easy to classify. Given the consistent performance across all three models, we selected the best model based on the highest average cross-validation accuracy.

After determining the best-performing model based on cross-validation, we retrained it on the full training and validation set. We then evaluated it on the held-out test set, which was not used during model selection or tuning. This provided an unbiased estimate of the model’s generalization performance.

Overall, our results show that even relatively simple neural networks are highly effective on this classification task. Increasing complexity provides marginal improvements, but care must be taken to avoid overfitting, especially on datasets that are already well-structured and easily separable. Cross-validation proved to be an effective method for comparing models and selecting the most appropriate one

## Before You Submit...

00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.